## Hybrid Retriever- Combining Dense And Sparse Retriever

In [11]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.schema import Document

In [12]:
model_name = "BAAI/bge-large-en-v1.5"
model_kwargs = {"device": "cpu"}

try:
    # Load from disk only if already present
    print(f"Searching for {model_name} locally...")
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name, model_kwargs={**model_kwargs, "local_files_only": True}
    )
    print("Model loaded from local storage.")

except Exception as e:
    # If not found, download it
    print("Model not found locally. Downloading from Hugging Face...")
    embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs=model_kwargs)
    print("Download complete and model initialized!")

Searching for BAAI/bge-large-en-v1.5 locally...
Model loaded from local storage.


In [13]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic ai application."),
    Document(page_content="Langchain has many types of retrievers."),
]

# Step 2: Dense Retriever (FAISS + HuggingFace)
embedding_model = embeddings
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever()

In [14]:
#sparse retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3  ##top- k documents to retriever

# step 4 : Combine with Ensemble Retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever], weight=[0.7, 0.3]
)

In [15]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000019FF35F5F40>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001A03C847D70>, k=3)], weights=[0.5, 0.5])

In [16]:
# Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
Langchain can be used to develop agentic ai application.

🔹 Document 3:
Langchain has many types of retrievers.

🔹 Document 4:
Pinecone is a vector database for semantic search.


### RAG Pipeline with hybrid retriever

In [17]:
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

In [18]:
# Step 5: Prompt Template
prompt = PromptTemplate.from_template(
    """
Answer the question based on the context below.

Context:
{context}

Question: {input}
"""
)

# Step 6: LLM initialization
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load environment variables
load_dotenv()

# Initialize LLM
llm = ChatGroq(model_name="openai/gpt-oss-20b")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A03DC11820>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A03DC13A10>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [19]:
# Create stuff Docuemnt Chain
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

# create Full rAg chain
rag_chain = create_retrieval_chain(
    retriever=hybrid_retriever, combine_docs_chain=document_chain
)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000019FF35F5F40>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x000001A03C847D70>, k=3)], weights=[0.5, 0.5]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {input}\n')
            | ChatGroq(profile={

In [20]:
# Step 9: Ask a question
query = {"input": "How can I build an app using LLMs?"}
response = rag_chain.invoke(query)

# Step 10: Output
print("✅ Answer:\n", response["answer"])

print("\n📄 Source Documents:")
for i, doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content}")

✅ Answer:
 **A quick‑start guide to building an LLM‑powered app with LangChain (and optional Pinecone for semantic search)**  

| Step | What to do | Why it matters | Typical code snippet |
|------|------------|----------------|----------------------|
| 1. **Choose an LLM provider** | OpenAI, Anthropic, Cohere, Gemini, etc. | Determines the model you’ll talk to. | ```python<br>from langchain.llms import OpenAI<br>llm = OpenAI(model_name="gpt-4o")<br>``` |
| 2. **Set up your environment** | Python 3.10+, virtualenv/conda, install LangChain. | Keeps dependencies isolated. | ```bash<br>python -m venv venv<br>source venv/bin/activate<br>pip install langchain[all]  # includes OpenAI, Pinecone, etc.<br>``` |
| 3. **Create a retriever (optional but powerful)** | • **Vector store**: Pinecone, Qdrant, Weaviate<br>• **Document store**: local PDFs, web pages, databases | Enables semantic search so the LLM can “look up” facts. | ```python<br>from langchain.vectorstores import Pinecone<br>from lang